In [1]:
!pip install streamlit pyngrok pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 88.3 MB/s eta 0:00:00


In [2]:
%%writefile app.py
import streamlit as st
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pandas as pd

from analytics_engine import *

st.set_page_config(layout="wide")
st.title("APL Logistics Profitability Dashboard")

DEFAULT_PATH = "APL_Logistics.csv"

df = None

if os.path.exists(DEFAULT_PATH):
    df = load_data(DEFAULT_PATH)
else:
    uploaded_file = st.sidebar.file_uploader("/content/APL_Logistics.csv", type=["csv"])
    if uploaded_file:
        df = load_data(uploaded_file)

if df is None:
    st.stop()

st.subheader("Data Preview")
st.dataframe(df.head())

if df.empty:
    st.error("Dataset empty")
    st.stop()


# FILTERS
# -----------------------------
def safe(col):
    return df[col].dropna().unique() if col in df.columns else []

segment = st.sidebar.multiselect("Segment", safe('Customer Segment'))
category = st.sidebar.multiselect("Category", safe('Category Name'))
market = st.sidebar.multiselect("Market", safe('Market'))
region = st.sidebar.multiselect("Region", safe('Order Region'))
product = st.sidebar.multiselect("Product", safe('Product Name'))

slider = st.sidebar.slider("Discount Simulation",0.0,1.0,0.2)

if segment: df = df[df['Customer Segment'].isin(segment)]
if category: df = df[df['Category Name'].isin(category)]
if market: df = df[df['Market'].isin(market)]
if region: df = df[df['Order Region'].isin(region)]
if product: df = df[df['Product Name'].isin(product)]

# KPI + EXECUTIVE SUMMARY
# -----------------------------
k = compute_kpis(df)

st.header("Executive Insights")

st.write(f"""
- Total Revenue: {k['Revenue']:.2f}
- Total Profit: {k['Profit']:.2f}
- Profit Margin: {k['Margin']:.2f}%
""")

gap = k['Revenue'] - k['Profit']
st.info(f"Revenue-Profit Gap: {gap:.2f}")

#Dynamic severity
if k['Margin'] < 5:
    st.error("Critical: Business operating at very low profitability")
elif k['Margin'] < 10:
    st.warning("Warning: Profit margins are under pressure")
else:
    st.success("Healthy profitability levels")

st.write("""
Key Observations:
- High revenue does not guarantee high profitability
- Discounts significantly impact margins
- Some customers and products drive disproportionate profit
""")

c1,c2,c3,c4 = st.columns(4)
c1.metric("Revenue",f"{k['Revenue']:.2f}")
c2.metric("Profit",f"{k['Profit']:.2f}")
c3.metric("Margin %",f"{k['Margin']:.2f}")
c4.metric("Discount Impact %",f"{k['Discount Impact Ratio']:.2f}")


# TREND
# -----------------------------
t = trend_analysis(df)
if t is not None:
    fig, ax = plt.subplots()
    ax.plot(t.iloc[:,0], t.iloc[:,1], label="Revenue")
    ax.plot(t.iloc[:,0], t.iloc[:,2], label="Profit")
    ax.legend()
    plt.xticks(rotation=45)
    st.pyplot(fig)

# CUSTOMER
# -----------------------------
cust = customer_analysis(df)
if not cust.empty:
    st.subheader("Customer Analysis")

    st.dataframe(cust.head(10))
    st.dataframe(cust.tail(10))

    #Profit concentration
    pc = profit_concentration(cust)
    st.info(f"Top 10 customers contribute {pc:.2f}% of total profit")

    loss_customers = cust[cust.iloc[:,2] < 0]
    if not loss_customers.empty:
        st.warning("Loss-making customers detected")
        st.dataframe(loss_customers)


# PRODUCT
# -----------------------------
prod, loss, risky = product_analysis(df)
if not prod.empty:
    st.subheader("Product Performance")

    st.dataframe(prod.head(10))

    if not loss.empty:
        st.warning("Loss-making products detected")
        st.dataframe(loss)

    if not risky.empty:
        st.warning("High revenue but low margin products detected")
        st.dataframe(risky)


# CATEGORY
# -----------------------------
cat, heat = category_analysis(df)

if not cat.empty:
    st.subheader("Category Performance")

    loss_cat = cat[cat['Margin (%)'] < 0]
    if not loss_cat.empty:
        st.warning("Loss-making categories detected")
        st.dataframe(loss_cat)

if heat is not None:
    fig, ax = plt.subplots()
    sns.heatmap(heat, ax=ax)
    st.pyplot(fig)


# MARKET
# -----------------------------
st.subheader("Market Performance")
m = market_analysis(df)
st.dataframe(m)

weak = m[
    (m.iloc[:,1] > m.iloc[:,1].quantile(0.75)) &
    (m['Margin (%)'] < 10)
]

if not weak.empty:
    st.warning("High revenue but low profit markets detected")
    st.dataframe(weak)

# DISCOUNT
# -----------------------------
disc = discount_analysis(df, slider)

if 'Discount Rate' in disc.columns:
    fig, ax = plt.subplots()
    ax.scatter(disc['Discount Rate'], disc['Profit Ratio'])
    st.pyplot(fig)

    avg_profit = df[get_profit_col(df)].mean()
    sim_profit = disc['Simulated Profit'].mean()

    st.info(f"Avg Profit Before Discount: {avg_profit:.2f}")
    st.info(f"Avg Profit After Simulation: {sim_profit:.2f}")

    threshold = disc[disc['Profit Ratio'] < 0]['Discount Rate'].mean()
    if pd.notna(threshold):
        st.warning(f"Discounts above {threshold:.2f} are likely causing losses")

    st.write("Higher discounts reduce margins and can lead to losses beyond threshold levels.")

# SEGMENT
# -----------------------------
seg = segment_contribution(df)
if not seg.empty:
    st.subheader("Customer Segment Contribution")
    st.bar_chart(seg)
    st.info("High-performing segments should be prioritized for growth")

# REGION / COUNTRY
# -----------------------------
st.subheader("Region Performance")
st.dataframe(region_analysis(df))

st.subheader("Country Performance")
st.dataframe(country_analysis(df))

Writing app.py


In [3]:
from pyngrok import ngrok
import os

# Add your auth token (IMPORTANT)
ngrok.set_auth_token("3DYz1LeXe4iod9UpuZTgSQdWOf0_65esX5w1Vf7UucXVCq6Xv")

# Kill previous sessions
!pkill -f streamlit

# Start Streamlit app
get_ipython().system_raw('streamlit run app.py &')

# Create tunnel
public_url = ngrok.connect(8501)

print("Streamlit App URL:", public_url)

Streamlit App URL: NgrokTunnel: "https://envious-spring-rust.ngrok-free.dev" -> "http://localhost:8501"
